asfasfasf

In [32]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
)
from sklearn.model_selection import train_test_split

In [33]:
PROJECT_ROOT = Path.cwd()

MODELS_DIR = PROJECT_ROOT / "../models"
DATA_PATH = PROJECT_ROOT / "../data" / "processed" / "nhanes_merged_adults_final.csv"

OUTPUT_DIR = PROJECT_ROOT / "../evaluation"
OUTPUT_DIR.mkdir(exist_ok=True)

print("MODELS_DIR:", MODELS_DIR)
print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

MODELS_DIR: c:\Users\Philipp\AIBootcamp\halfFull\notebooks\..\models
DATA_PATH: c:\Users\Philipp\AIBootcamp\halfFull\notebooks\..\data\processed\nhanes_merged_adults_final.csv
OUTPUT_DIR: c:\Users\Philipp\AIBootcamp\halfFull\notebooks\..\evaluation


In [34]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(7437, 878)


C:\Users\Philipp\AppData\Local\Temp\ipykernel_41056\1549940479.py:1: DtypeWarning: Columns (0: medication_17, 1: medication_18, 2: medication_19, 3: medication_20, 4: medication_21, 5: medication_22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,SEQN,age_years,income_poverty_ratio,mec_exam_weight,interview_weight,survey_psu,survey_stratum,gender,ethnicity,education,...,iron_deficiency,hepatic_insufficiency,electrolyte_imbalance,infection_inflammation,CFS_suspect,myalgia,anxiety,depression,diabetes,prediabetes
0,109266.0,29.0,5.00,8.154968e+03,7825.646112,2.0,168.0,Female,Non-Hispanic Asian,College graduate or above,...,1,0,0,0.0,0,0,0,0,0,1
1,109267.0,21.0,5.00,5.397605e-79,26379.991724,1.0,156.0,Female,Other Hispanic,Some college / AA,...,0,0,0,0.0,0,0,0,0,0,0
2,109268.0,18.0,1.66,5.397605e-79,19639.221008,1.0,155.0,Female,Non-Hispanic White,NaN,...,0,0,0,0.0,0,0,0,0,0,0
3,109271.0,49.0,NaN,8.658733e+03,8481.589837,1.0,167.0,Male,Non-Hispanic White,9-11th grade,...,0,0,0,1.0,0,0,0,0,0,0
4,109273.0,36.0,0.83,2.216360e+04,20171.847767,1.0,155.0,Male,Non-Hispanic White,Some college / AA,...,0,0,0,0.0,0,0,0,0,0,0


In [35]:
def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def find_model_files(models_dir: Path):
    """
    Sucht alle .joblib Modelle und versucht die passende metadata.json zuzuordnen.
    Erwartet bevorzugt:
      modelname.joblib
      modelname_metadata.json
    oder alternativ:
      modelname.joblib
      modelname.metadata.json
    """
    model_entries = []

    for model_path in models_dir.glob("*.joblib"):
        stem = model_path.stem

        candidate_meta_1 = models_dir / f"{stem}_metadata.json"
        candidate_meta_2 = models_dir / f"{stem}.metadata.json"

        metadata_path = None
        if candidate_meta_1.exists():
            metadata_path = candidate_meta_1
        elif candidate_meta_2.exists():
            metadata_path = candidate_meta_2

        model_entries.append({
            "model_name": stem,
            "model_path": model_path,
            "metadata_path": metadata_path
        })

    return sorted(model_entries, key=lambda x: x["model_name"])

In [36]:
model_entries = find_model_files(MODELS_DIR)

pd.DataFrame(model_entries)

,model_name,model_path,metadata_path
0,anemia_combined_lr,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
1,electrolyte_imbalance_compact_quiz_demo_med_sc...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
2,hepatitis_lr_l1_34feat,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
3,hepatitis_rf_cal_33feat,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
4,inflammation_lr_l2_32feat,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
5,iron_deficiency_checkup_lr,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
6,kidney_lr_l2_routine_30feat,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
7,liver_lr_l2_13feat,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
8,perimenopause_gradient_boosting,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...
9,prediabetes_focused_quiz_demo_med_screening_la...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...,c:\Users\Philipp\AIBootcamp\halfFull\notebooks...


In [37]:
def extract_target_and_features(meta: dict, model_name: str = ""):
    """
    Liest target und features robust aus unterschiedlich strukturierten metadata.json Dateien.
    """

    # ---------- TARGET ----------
    possible_target_keys = [
        "target",
        "target_column",
        "label",
        "y_col"
    ]

    target = None
    for key in possible_target_keys:
        if key in meta:
            target = meta[key]
            break

    # Fallback: aus Modellnamen ableiten
    if target is None:
        fallback_targets = {
            "thyroid": "thyroid",
            "kidney": "kidney_disease",
            "liver": "liver_disease",
        }
        for k, v in fallback_targets.items():
            if k in model_name.lower():
                target = v
                break

    if target is None:
        raise KeyError(
            f"Kein Target-Key gefunden. Verfügbare Keys: {list(meta.keys())}"
        )

    # ---------- FEATURES ----------
    features = None

    if "features" in meta:
        features = meta["features"]
    elif "feature_names" in meta:
        features = meta["feature_names"]
    elif "selected_features" in meta:
        features = meta["selected_features"]
    elif "x_cols" in meta:
        features = meta["x_cols"]
    elif "all_features" in meta:
        features = meta["all_features"]
    else:
        # Falls Features auf mehrere Listen verteilt sind
        combined = []
        for key in ["base_features", "lab_features", "quest_features", "miss_flag_features"]:
            if key in meta and isinstance(meta[key], list):
                combined.extend(meta[key])

        if combined:
            # Reihenfolge beibehalten, Duplikate entfernen
            seen = set()
            features = []
            for f in combined:
                if f not in seen:
                    seen.add(f)
                    features.append(f)

    if features is None:
        raise KeyError(
            f"Kein Feature-Key gefunden. Verfügbare Keys: {list(meta.keys())}"
        )

    if not isinstance(features, list):
        raise TypeError(f"Features müssen eine Liste sein, sind aber {type(features)}")

    return target, features

In [38]:
def basic_target_cleanup(y: pd.Series) -> pd.Series:
    """
    Macht aus dem Target eine saubere binäre 0/1 Series.
    Nur simpel gehalten. Falls ihr NHANES-Sondercodes habt, hier erweitern.
    """
    y = y.copy()

    # typische Fälle
    if y.dtype == "bool":
        return y.astype(int)

    # numerisch
    if pd.api.types.is_numeric_dtype(y):
        y = y.dropna()
        return y.astype(int)

    # Strings
    mapping = {
        "yes": 1, "no": 0,
        "true": 1, "false": 0,
        "positive": 1, "negative": 0
    }
    y = y.astype(str).str.strip().str.lower().map(mapping)
    return y

In [39]:
def evaluate_single_model(
    df: pd.DataFrame,
    model_path: Path,
    metadata_path: Path | None,
    test_size: float = 0.2,
    random_state: int = 42,
):
    """
    Lädt target/features bevorzugt aus dem Joblib-Wrapper, sonst aus metadata.json.
    Berechnet Brier Score, ROC-AUC und Average Precision auf einem reproduzierbaren Holdout-Split.
    """
    model_obj = joblib.load(model_path)

    # 1) Echtes Modell aus Wrapper ziehen
    joblib_target = None
    joblib_features = None
    wrapper_threshold = None

    if isinstance(model_obj, dict):
        joblib_target = model_obj.get("target")
        joblib_features = model_obj.get("features")
        wrapper_threshold = model_obj.get("threshold")

        if "model" in model_obj:
            model = model_obj["model"]
        elif "estimator" in model_obj:
            model = model_obj["estimator"]
        elif "classifier" in model_obj:
            model = model_obj["classifier"]
        else:
            raise AttributeError(
                f"Modell {model_path.stem} ist ein dict, aber kein Modell-Key gefunden. Keys: {list(model_obj.keys())}"
            )
    else:
        model = model_obj

    # 2) target / features bestimmen
    if joblib_target is not None and joblib_features is not None:
        target = joblib_target
        features = joblib_features
    else:
        if metadata_path is None or not metadata_path.exists():
            raise FileNotFoundError(f"Keine metadata.json gefunden für {model_path.name}")
        meta = load_json(metadata_path)
        target, features = extract_target_and_features(meta, model_path.stem)

    # 3) Sonderfälle / Target-Aliase
    target_aliases = {
        "kidney_disease": ["kidney_disease", "weak_failing_kidneys", "kiq022_weak_failing_kidneys"],
        "liver_disease": ["liver_disease", "mcq160l_liver_condition", "liver_condition"],
        "thyroid": ["thyroid"],
        "perimenopause_proxy": ["perimenopause_proxy"],
    }
    if target not in df.columns and target in target_aliases:
        for alt in target_aliases[target]:
            if alt in df.columns:
                target = alt
                break

    if target not in df.columns:
        raise ValueError(f"Target '{target}' fehlt im DataFrame für Modell {model_path.stem}")

    missing_features = [col for col in features if col not in df.columns]
    if missing_features:
        raise ValueError(
            f"Für Modell {model_path.stem} fehlen Features im DataFrame: {missing_features}"
        )

    work_df = df[features + [target]].copy()
    work_df = work_df.dropna(subset=[target])

    y = basic_target_cleanup(work_df[target])
    X = work_df[features]

    valid_idx = y.dropna().index
    X = X.loc[valid_idx]
    y = y.loc[valid_idx].astype(int)

    unique_vals = sorted(pd.Series(y).dropna().unique().tolist())
    if unique_vals not in ([0, 1], [0], [1]):
        raise ValueError(
            f"Target {target} in {model_path.stem} ist nicht sauber binär. Unique values: {unique_vals}"
        )
    if len(unique_vals) < 2:
        raise ValueError(
            f"Target {target} in {model_path.stem} hat im verfügbaren Datensatz nur eine Klasse: {unique_vals}"
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # 4) Wahrscheinlichkeiten extrahieren
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        y_proba = 1 / (1 + np.exp(-scores))
    else:
        raise AttributeError(
            f"Inneres Modell von {model_path.stem} hat weder predict_proba() noch decision_function()"
        )

    result = {
        "model_name": model_path.stem,
        "target": target,
        "n_rows_total": int(len(work_df)),
        "n_test": int(len(y_test)),
        "positive_rate_test": float(y_test.mean()),
        "brier_score": float(brier_score_loss(y_test, y_proba)),
        "roc_auc": float(roc_auc_score(y_test, y_proba)),
        "avg_precision": float(average_precision_score(y_test, y_proba)),
        "n_features": int(len(features)),
        "wrapper_threshold": wrapper_threshold,
        "features": features,
    }

    return result, y_test, y_proba, X_test

In [40]:
def prepare_evaluation_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def add_alias(new_col, candidates):
        if new_col in df.columns:
            return
        for c in candidates:
            if c in df.columns:
                df[new_col] = df[c]
                return

    # --------------------------------------------------
    # 1) Blutdruck-/Puls-Mittelwerte
    # --------------------------------------------------
    sbp_candidates = [c for c in [
        "sbp_1", "sbp_2", "sbp_3",
        "bpq150a_systolic_1", "bpq150b_systolic_2", "bpq150c_systolic_3"
    ] if c in df.columns]

    dbp_candidates = [c for c in [
        "dbp_1", "dbp_2", "dbp_3",
        "bpq160a_diastolic_1", "bpq160b_diastolic_2", "bpq160c_diastolic_3"
    ] if c in df.columns]

    pulse_candidates = [c for c in ["pulse_1", "pulse_2", "pulse_3"] if c in df.columns]

    if "sbp_mean" not in df.columns and sbp_candidates:
        df["sbp_mean"] = df[sbp_candidates].mean(axis=1)

    if "dbp_mean" not in df.columns and dbp_candidates:
        df["dbp_mean"] = df[dbp_candidates].mean(axis=1)

    if "pulse_mean" not in df.columns and pulse_candidates:
        df["pulse_mean"] = df[pulse_candidates].mean(axis=1)

    # --------------------------------------------------
    # 2) Geschlecht / Dummies
    # --------------------------------------------------
    if "gender_female" not in df.columns:
        if "gender" in df.columns:
            g = df["gender"].astype(str).str.lower().str.strip()
            df["gender_female"] = g.isin(["female", "f", "2"]).astype(int)
        elif "sex" in df.columns:
            g = df["sex"].astype(str).str.lower().str.strip()
            df["gender_female"] = g.isin(["female", "f", "2"]).astype(int)
        elif "riagendr" in df.columns:
            df["gender_female"] = pd.to_numeric(df["riagendr"], errors="coerce").eq(2).astype(int)

    # --------------------------------------------------
    # 3) Alias-Spalten aus mehreren Modellen
    # --------------------------------------------------
    alias_map = {
        "total_cholesterol": ["total_cholesterol_mg_dl"],
        "ldl_cholesterol": ["ldl_cholesterol_mg_dl"],
        "hdl_cholesterol": ["hdl_cholesterol_mg_dl"],
        "cholesterol": ["total_cholesterol_mg_dl"],
        "hdl": ["hdl_cholesterol_mg_dl"],
        "triglycerides": ["triglycerides_mg_dl"],

        "serum_glucose": ["serum_glucose_mg_dl", "glucose_mg_dl"],
        "fasting_glucose": ["fasting_glucose_mg_dl", "glucose_mg_dl"],
        "glucose": ["fasting_glucose_mg_dl", "glucose_mg_dl"],

        "creatinine": ["serum_creatinine", "serum_creatinine_mg_dl", "serum_creatinine_mg_dl_x"],
        "calcium": ["serum_calcium_mg_dl", "calcium_mg_dl"],
        "wbc": ["white_blood_cell_count", "wbc_count"],
        "total_protein": ["total_protein_g_dl"],

        "avg_drinks_per_day": ["alq130_avg_drinks_per_day"],
        "ever_heavy_drinker": ["alq151_ever_heavy_drinker"],
        "ever_heavy_drinker_daily": ["alq151_ever_heavy_drinker"],
        "blood_transfusion": ["mcq220_blood_transfusion", "ever_had_blood_transfusion"],
        "smoked_100_cigs": ["smq020_smoked_100_cigarettes", "smq020_smoked_100_cigs"],
        "avg_cigarettes_per_day": ["smd650_cigarettes_per_day", "cigarettes_per_day"],
        "cigarettes_per_day": ["smd650_cigarettes_per_day"],
        "smoking_now": ["smq040_smoke_now"],

        "sedentary_minutes": ["pad680_sedentary_minutes"],
        "moderate_exercise": ["pad615_moderate_recreational"],
        "vigorous_exercise": ["pad645_vigorous_recreational"],
        "moderate_recreational": ["pad615_moderate_recreational"],

        "sleep_hours_weekdays": ["sld012_sleep_hours_weekday"],
        "sleep_hours_weekday": ["sld012_sleep_hours_weekday"],
        "sleep_hours_weekend": ["sld013_sleep_hours_weekend"],
        "told_dr_trouble_sleeping": ["slq050_told_trouble_sleeping"],
        "trouble_sleeping": ["slq050_told_trouble_sleeping"],

        "general_health": ["huq010_general_health"],
        "general_health_condition": ["huq010_general_health"],
        "doctor_said_overweight": ["whq070_doctor_said_overweight"],
        "dr_said_reduce_fat": ["whq090_dr_said_reduce_fat"],
        "tried_to_lose_weight": ["whq110_tried_to_lose_weight"],

        "liver_condition": ["mcq160l_liver_condition"],
        "kidney_disease": ["kiq022_weak_failing_kidneys"],
        "ever_had_kidney_stones": ["kiq026_kidney_stones"],
        "ever_hepatitis_c": ["heq010_hepatitis_c"],

        "times_urinate_in_night": ["kiq480_nighttime_urination"],
        "how_often_urinary_leakage": ["kiq430_urine_leak_frequency"],
        "urinated_before_toilet": ["kiq421_urinate_before_toilet"],

        "taking_bp_prescription": ["bpq100d_bp_medication"],
        "ever_told_high_bp": ["bpq020_high_bp"],
        "told_high_cholesterol": ["bpq080_high_cholesterol"],
        "ever_told_high_cholesterol": ["bpq080_high_cholesterol"],

        "ever_told_heart_failure": ["mcq160b_congestive_heart_failure"],
        "heart_failure": ["mcq160b_congestive_heart_failure"],
        "ever_told_heart_attack": ["mcq160e_heart_attack"],
        "ever_told_stroke": ["mcq160f_stroke"],
        "ever_told_diabetes": ["diq010_diabetes"],
        "takes_diabetes_pills": ["diq070_diabetes_pills"],
        "taking_diabetic_pills": ["diq070_diabetes_pills"],
        "taking_insulin": ["diq050_insulin"],

        "taking_anemia_treatment": ["anq100_taking_anemia_treatment"],

        "regular_periods": ["rhq031_regular_periods"],
        "age_last_period": ["rhq060_age_last_period"],
        "ever_hormones": ["rhq540_ever_hormones"],

        "feeling_tired_little_energy": ["dpq040_tired_little_energy"],
        "sob_stairs": ["cdq010_sob_stairs"],
        "saw_dr_for_pain": ["paq_medical_help_pain"],
        "abdominal_pain": ["abd_pain"],
        "overnight_hospital": ["huq090_over_night_hospital"],
        "hospitalized_lastyear": ["huq090_over_night_hospital"],

        "work_schedule": ["ocq265_work_schedule"],
        "overall_work_schedule": ["ocq265_work_schedule"],
        "hours_worked_per_week": ["ocq180_hours_worked"],
        "times_healthcare_past_year": ["huq030_healthcare_times"],

        # Umgekehrte Alias-Richtung für Modelle, die Original-NHANES-Namen erwarten
        "huq010_general_health": ["general_health", "general_health_condition"],
        "dpq040_tired_little_energy": ["feeling_tired_little_energy"],
        "cdq010_sob_stairs": ["sob_stairs"],
        "sld012_sleep_hours_weekday": ["sleep_hours_weekdays"],
        "sld013_sleep_hours_weekend": ["sleep_hours_weekend"],
        "slq050_told_trouble_sleeping": ["told_dr_trouble_sleeping", "trouble_sleeping"],
        "pad680_sedentary_minutes": ["sedentary_minutes"],
        "rhq031_regular_periods": ["regular_periods"],
        "rhq060_age_last_period": ["age_last_period"],
        "rhq540_ever_hormones": ["ever_hormones"],
    }

    alias_map.update({
        # anemia / iron deficiency
        "ldl_cholesterol_mg_dl": ["ldl_cholesterol", "ldl"],
        "dpq040_tired_little_energy": ["feeling_tired_little_energy"],
        "cdq010_sob_stairs": ["sob_stairs"],
        "sld012_sleep_hours_weekday": ["sleep_hours_weekdays", "sleep_hours_weekday"],
        "sld013_sleep_hours_weekend": ["sleep_hours_weekend"],
        "slq050_told_trouble_sleeping": ["told_dr_trouble_sleeping", "trouble_sleeping"],
        "pad680_sedentary_minutes": ["sedentary_minutes"],
        "rhq060_age_last_period": ["age_last_period"],
        "rhq540_ever_hormones": ["ever_hormones"],

        # hepatitis
        "total_protein": ["total_protein_g_dl"],
        "wbc": ["white_blood_cell_count", "wbc_count"],
        "cholesterol": ["total_cholesterol_mg_dl", "total_cholesterol"],
        "triglycerides": ["triglycerides_mg_dl"],
        "hdl": ["hdl_cholesterol_mg_dl", "hdl_cholesterol"],
        "glucose": ["fasting_glucose_mg_dl", "serum_glucose", "glucose_mg_dl"],
        "general_health": ["huq010_general_health", "general_health_condition"],
        "education": ["education_level"],
        "bmi": ["body_mass_index"],

        # inflammation
        "total_cholesterol": ["total_cholesterol_mg_dl"],
        "ldl_cholesterol": ["ldl_cholesterol_mg_dl"],
        "hdl_cholesterol": ["hdl_cholesterol_mg_dl"],
        "serum_glucose": ["serum_glucose_mg_dl", "glucose_mg_dl"],
        "fasting_glucose": ["fasting_glucose_mg_dl"],
        "creatinine": ["serum_creatinine", "serum_creatinine_mg_dl"],
        "sedentary_minutes": ["pad680_sedentary_minutes"],
        "sleep_hours_weekdays": ["sld012_sleep_hours_weekday"],
        "regular_periods": ["rhq031_regular_periods"],

        # kidney
        "hdl_cholesterol": ["hdl_cholesterol_mg_dl"],
        "ever_told_high_bp": ["bpq020_high_bp"],
        "taking_bp_prescription": ["bpq100d_bp_medication"],
        "told_high_cholesterol": ["bpq080_high_cholesterol"],
        "ever_told_heart_failure": ["mcq160b_congestive_heart_failure"],
        "ever_told_heart_attack": ["mcq160e_heart_attack"],
        "ever_told_stroke": ["mcq160f_stroke"],
        "ever_told_diabetes": ["diq010_diabetes"],
        "times_urinate_in_night": ["kiq480_nighttime_urination"],
        "how_often_urinary_leakage": ["kiq430_urine_leak_frequency"],
        "urinated_before_toilet": ["kiq421_urinate_before_toilet"],
        "ever_had_kidney_stones": ["kiq026_kidney_stones"],
        "ever_had_blood_transfusion": ["mcq220_blood_transfusion", "blood_transfusion"],
        "ever_told_arthritis": ["mcq160a_arthritis"],
        "times_healthcare_past_year": ["huq030_healthcare_times"],
        "moderate_recreational": ["pad615_moderate_recreational"],

        # liver
        "takes_diabetes_pills": ["diq070_diabetes_pills", "taking_diabetic_pills"],
        "ever_heavy_drinker_daily": ["alq151_ever_heavy_drinker", "ever_heavy_drinker"],
        "dr_said_reduce_fat": ["whq090_dr_said_reduce_fat"],
        "saw_dr_for_pain": ["paq_medical_help_pain"],
        "abdominal_pain": ["abd_pain"],
        "overnight_hospital": ["huq090_over_night_hospital"],
        "ever_hepatitis_c": ["heq010_hepatitis_c"],

        # thyroid
        "general_health_condition": ["huq010_general_health", "general_health"],
        "tried_to_lose_weight": ["whq110_tried_to_lose_weight"],
        "avg_cigarettes_per_day": ["smd650_cigarettes_per_day", "cigarettes_per_day"],
        "overall_work_schedule": ["ocq265_work_schedule", "work_schedule"],
        "sleep_hours_weekdays": ["sld012_sleep_hours_weekday"],
        "ever_told_high_cholesterol": ["bpq080_high_cholesterol"],
    })

    for new_col, candidates in alias_map.items():
        add_alias(new_col, candidates)

    # --------------------------------------------------
    # 4) Alkohol-Risikosignal
    # --------------------------------------------------
    if "alcohol_any_risk_signal" not in df.columns:
        parts = []
        if "avg_drinks_per_day" in df.columns:
            parts.append(pd.to_numeric(df["avg_drinks_per_day"], errors="coerce").fillna(0) > 2)
        if "ever_heavy_drinker" in df.columns:
            parts.append(pd.to_numeric(df["ever_heavy_drinker"], errors="coerce").fillna(0) == 1)

        if parts:
            signal = parts[0]
            for p in parts[1:]:
                signal = signal | p
            df["alcohol_any_risk_signal"] = signal.astype(int)

    # explizite Zusatzfeatures, falls Rohspalten existieren
    if "alcohol_any_risk_signal" not in df.columns:
        parts = []

        if "avg_drinks_per_day" in df.columns:
            parts.append(pd.to_numeric(df["avg_drinks_per_day"], errors="coerce").fillna(0) > 2)

        if "ever_heavy_drinker" in df.columns:
            parts.append(pd.to_numeric(df["ever_heavy_drinker"], errors="coerce").fillna(0) == 1)

        if parts:
            signal = parts[0]
            for p in parts[1:]:
                signal = signal | p
            df["alcohol_any_risk_signal"] = signal.astype(int)

    if "gender_female" not in df.columns and "gender" in df.columns:
        g = df["gender"].astype(str).str.lower().str.strip()
        df["gender_female"] = g.isin(["female", "f", "2"]).astype(int)

    if "gender_female" not in df.columns and "sex" in df.columns:
        g = df["sex"].astype(str).str.lower().str.strip()
        df["gender_female"] = g.isin(["female", "f", "2"]).astype(int)


    # --------------------------------------------------
    # 5) Missing-Flags effizient
    # --------------------------------------------------
    missing_flag_dict = {}
    for col in df.columns:
        miss_col = f"{col}_miss"
        if miss_col not in df.columns:
            missing_flag_dict[miss_col] = df[col].isna().astype(int)

    if missing_flag_dict:
        miss_df = pd.DataFrame(missing_flag_dict, index=df.index)
        df = pd.concat([df, miss_df], axis=1)

    return df.copy()

In [41]:
df_eval = prepare_evaluation_dataframe(df)
print(df_eval.shape)

(7437, 1780)


## Evaluation ausführen

Diese Version enthält jetzt:
- eine definierte `evaluate_single_model(...)` Funktion
- Support für Joblib-Wrapper-Dicts mit `{"model": ...}`
- robustere Feature-/Target-Erkennung
- erweitertes Feature-Aliasing für eure historischen Modellnamen

Falls einzelne Modelle weiterhin scheitern, sind das voraussichtlich echte Sonderfälle im Trainings-Setup und nicht mehr Notebook-Strukturfehler.


In [42]:
all_results = []
errors = []

for entry in model_entries:
    try:
        result, y_test, y_proba, X_test = evaluate_single_model(
            df=df_eval,
            model_path=entry["model_path"],
            metadata_path=entry["metadata_path"],
            test_size=0.2,
            random_state=42,
        )
        all_results.append(result)
        print(f"OK: {entry['model_name']} | Brier={result['brier_score']:.4f}")

    except Exception as e:
        errors.append({
            "model_name": entry["model_name"],
            "error": str(e)
        })
        print(f"ERROR: {entry['model_name']} -> {e}")

ERROR: anemia_combined_lr -> Für Modell anemia_combined_lr fehlen Features im DataFrame: ['ldl_cholesterol_mg_dl', 'dpq040_tired_little_energy', 'huq010_general_health', 'cdq010_sob_stairs', 'sld012_sleep_hours_weekday', 'sld013_sleep_hours_weekend', 'slq050_told_trouble_sleeping', 'pad680_sedentary_minutes', 'rhq031_regular_periods', 'rhq060_age_last_period', 'rhq540_ever_hormones']
ERROR: electrolyte_imbalance_compact_quiz_demo_med_screening_labs_threshold_05 -> Für Modell electrolyte_imbalance_compact_quiz_demo_med_screening_labs_threshold_05 fehlen Features im DataFrame: ['alcohol_any_risk_signal']
ERROR: hepatitis_lr_l1_34feat -> Für Modell hepatitis_lr_l1_34feat fehlen Features im DataFrame: ['total_protein', 'wbc', 'avg_drinks_per_day', 'ever_heavy_drinker', 'blood_transfusion', 'smoked_100_cigs', 'liver_condition', 'general_health', 'hospitalized_lastyear', 'total_protein_miss', 'wbc_miss', 'avg_drinks_per_day_miss', 'ever_heavy_drinker_miss', 'liver_condition_miss']
ERROR: hep

# Checking the joblib files

In [43]:
def inspect_model_object(model_path: Path):
    obj = joblib.load(model_path)
    print("MODEL:", model_path.stem)
    print("TYPE:", type(obj))

    if hasattr(obj, "keys"):
        try:
            print("DICT KEYS:", list(obj.keys()))
        except Exception:
            pass

    print("has predict:", hasattr(obj, "predict"))
    print("has predict_proba:", hasattr(obj, "predict_proba"))
    print("has decision_function:", hasattr(obj, "decision_function"))
    print("-" * 60)

    return obj

In [44]:
inspect_model_object(MODELS_DIR / "prediabetes_focused_quiz_demo_med_screening_labs_threshold_045.joblib")
inspect_model_object(MODELS_DIR / "sleep_disorder_compact_quiz_demo_med_screening_labs_threshold_04.joblib")

MODEL: prediabetes_focused_quiz_demo_med_screening_labs_threshold_045
TYPE: <class 'dict'>
DICT KEYS: ['model', 'threshold', 'features', 'model_name', 'target']
has predict: False
has predict_proba: False
has decision_function: False
------------------------------------------------------------
MODEL: sleep_disorder_compact_quiz_demo_med_screening_labs_threshold_04
TYPE: <class 'dict'>
DICT KEYS: ['model', 'threshold', 'features', 'model_name', 'target', 'disease']
has predict: False
has predict_proba: False
has decision_function: False
------------------------------------------------------------


{'model': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(add_indicator=True,
                                                                                  strategy='median')),
                                                                   ('scaler',
                                                                    StandardScaler())]),
                                                   ['sld012___sleep_hours___weekdays_or_workdays',
                                                    'sld013___sleep_hours___weekends',
                                                    'age_years', 'med_count',
                                                    'fasting_glucose_mg_dl',
                                                    'hdl_cholesterol_mg_dl',
                              

In [45]:
pred_obj = joblib.load(MODELS_DIR / "prediabetes_focused_quiz_demo_med_screening_labs_threshold_045.joblib")
sleep_obj = joblib.load(MODELS_DIR / "sleep_disorder_compact_quiz_demo_med_screening_labs_threshold_04.joblib")

print(type(pred_obj["model"]))
print(type(sleep_obj["model"]))

<class 'sklearn.pipeline.Pipeline'>
<class 'sklearn.pipeline.Pipeline'>


In [46]:
print("pred has predict_proba:", hasattr(pred_obj["model"], "predict_proba"))
print("sleep has predict_proba:", hasattr(sleep_obj["model"], "predict_proba"))

pred has predict_proba: True
sleep has predict_proba: True


In [47]:
len(all_results), len(errors)

(2, 10)

In [48]:
pd.DataFrame(errors)

,model_name,error
0,anemia_combined_lr,Für Modell anemia_combined_lr fehlen Features ...
1,electrolyte_imbalance_compact_quiz_demo_med_sc...,Für Modell electrolyte_imbalance_compact_quiz_...
2,hepatitis_lr_l1_34feat,Für Modell hepatitis_lr_l1_34feat fehlen Featu...
3,hepatitis_rf_cal_33feat,Für Modell hepatitis_rf_cal_33feat fehlen Feat...
4,inflammation_lr_l2_32feat,Für Modell inflammation_lr_l2_32feat fehlen Fe...
5,iron_deficiency_checkup_lr,Für Modell iron_deficiency_checkup_lr fehlen F...
6,kidney_lr_l2_routine_30feat,Target 'kidney_disease' fehlt im DataFrame für...
7,liver_lr_l2_13feat,Für Modell liver_lr_l2_13feat fehlen Features ...
8,perimenopause_gradient_boosting,Target 'perimenopause_proxy' fehlt im DataFram...
9,thyroid_lr_l2_18feat,Für Modell thyroid_lr_l2_18feat fehlen Feature...
